In [ ]:
# AutoDIAL Full-Mamba HADA no-IM-TTA — BCI IV-2a LOSO (one cell). Enable GPU + Internet.
import codecs
import os
import re
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

# ---------------- Job configuration ----------------
# 'autodial' = learned source/target statistic mixing (main method);
# 'none'     = original HADA without AutoDIAL, for a same-branch baseline.
FEATURE_ALIGNMENT = 'autodial'
TARGET_SUBJECTS = list(range(1, 10))
JOB_TAG = 'no_im_tta_s01_s09'

DATASET = 'bcic2a'
BRANCH = 'feature/hada-autodial-zhou-mi-preprocessing-no-im-tta'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
RUN_TAG = f"{DATASET}_{FEATURE_ALIGNMENT}_{JOB_TAG}"
REPO_PATH = Path(f"/kaggle/working/tcformer-autodial-{DATASET}")
MNE_DATA = Path("/kaggle/working/mne_data")
TRAIN_LOG = Path(f"/kaggle/working/autodial_{RUN_TAG}.log")
RESULT_ARCHIVE = Path(f"/kaggle/working/autodial_{RUN_TAG}_results")


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print("+", " ".join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    started_at = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end="", flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f"[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n"
                print(heartbeat, end="", flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break

        remaining = decoder.decode(b"", final=True)
        if remaining:
            print(remaining, end="", flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Training exited with code {return_code}. Full log: {log_path}"
        )


# ---------------- Clone branch and verify the implementation ----------------
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
if (REPO_PATH / ".git").is_dir():
    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_PATH)
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_PATH)
    # A previous run edits the YAML locally; reset so reruns are safe.
    run(["git", "checkout", "--force", BRANCH], cwd=REPO_PATH)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_PATH])
run(["git", "log", "-1", "--oneline"], cwd=REPO_PATH)

pipeline_text = (REPO_PATH / "train_pipeline.py").read_text(encoding="utf-8")
model_text = (REPO_PATH / "models/hada_tcformer.py").read_text(encoding="utf-8")
datamodule_text = (REPO_PATH / "utils/get_datamodule_cls.py").read_text(encoding="utf-8")
assert "ModelCheckpoint" in pipeline_text and 'monitor="val_acc"' in pipeline_text
assert "class AutoDIALFeatureNorm" in model_text
assert "zhou2016_loso" in datamodule_text
print("Verified: AutoDIAL model, Zhou2016 support, best source-validation checkpoint.")

UV = shutil.which("uv") or "uv"
run([UV, "venv", "--clear", "--python", "3.10", ".venv"], cwd=REPO_PATH)
PYTHON = REPO_PATH / ".venv/bin/python"
run([UV, "pip", "install", "--python", PYTHON, "torch==2.7.1", "torchvision==0.22.1",
     "--index-url", "https://download.pytorch.org/whl/cu126"])
run([UV, "pip", "install", "--python", PYTHON, "-r", "requirements.txt"], cwd=REPO_PATH)

# ---------------- Config override (subjects, alignment mode, compact logs) -------
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    "c['log_every_n_epochs']=10; "
    "c['enable_progress_bar']=False; "
    "c['model_kwargs']['log_every_n_batches']=0; "
    f"c['model_kwargs']['feature_alignment']={FEATURE_ALIGNMENT!r}; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, "-c", override], cwd=REPO_PATH)
run(["nvidia-smi"])
run([PYTHON, "-c", "import torch; print('PyTorch:', torch.__version__); "
     "print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
     "print('GPU:', torch.cuda.get_device_name(0))"])

# ---------------- Train / test ----------------
MNE_DATA.mkdir(parents=True, exist_ok=True)
results_dir = REPO_PATH / "results"
if results_dir.exists():
    # A rerun in the same session must not mix old and new result folders.
    shutil.rmtree(results_dir)
environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "MPLBACKEND": "Agg",
    "MNE_DATA": str(MNE_DATA),
    "MNE_DATASETS_BNCI_PATH": str(MNE_DATA),
})
print(f"===== AUTODIAL FULL-MAMBA HADA NO-IM-TTA | BCI IV-2a | 9 SUBJECTS | 4 CLASSES | 22 CH | alignment={FEATURE_ALIGNMENT} "
      f"| TARGETS={TARGET_SUBJECTS} =====", flush=True)
print("Protocol: Source subjects: session T trains, session E validates (best checkpoint). Target: session T is unlabeled UDA data; session E is used for AutoDIAL recalibration and primary testing without IM-TTA.", flush=True)
print("Live log:", TRAIN_LOG, flush=True)
run([PYTHON, "-u", "train_pipeline.py", "--model", "hada_tcformer", "--dataset", DATASET,
     "--loso", "--gpu_id", "0"], cwd=REPO_PATH, env=environment, stream=True,
    log_path=TRAIN_LOG)

# ---------------- Summary: learned AutoDIAL alpha per target subject ----------------
log_text = TRAIN_LOG.read_text(encoding="utf-8", errors="replace")
alpha_prefix = "AutoDIAL alpha per feature group: "
alphas = [line.split(alpha_prefix, 1)[1].strip() for line in log_text.splitlines() if alpha_prefix in line]
for subject_id, alpha in zip(TARGET_SUBJECTS, alphas):
    print(f"S{subject_id:02d} final alpha (CNN groups 1-3, SSM group): {alpha}")

archive = shutil.make_archive(str(RESULT_ARCHIVE), "zip", root_dir=REPO_PATH, base_dir="results")
print("Completed targets:", TARGET_SUBJECTS)
print("Kaggle output archive:", archive)
print("Kaggle output log:", TRAIN_LOG)
